# Tarea Práctica: Mecanismos de Verificación y Control de Integridad

## Contexto
Estás al mando de la plataforma de datos de una gran red de logística. En el sistema distribuido intervienen camiones, almacenes, una pasarela de pagos y el Data Lake central.
Como vimos en la **UT3 - Capítulo 3**, no puedes simplemente "confiar" en que los datos viajan bien. Necesitas **verificar**, establecer **controles de frontera** y aplicar mecánicas de **cuarentena** y **reconciliación** constantes.

## Objetivo
Deberás auditar tres conjuntos de datos que acaban de llegar a la capa de ingesta, aplicando los conceptos técnicos de la teoría.

Debes abordar los siguientes puntos de control:
1.  **Validación de Frontera y Cuarentena:** Filtrar transacciones imposibles (cantidades o importes negativos, nulos) ANTES de que entren a la zona Silver, aislando los errores.
2.  **Verificación por Hash (Checksums):** Detectar corrupción de "man in the middle" o fallos de disco comprobando que el hash del archivo original corresponde con los datos recibidos.
3.  **Auditoría de Reconciliación:** Cruzar los datos del almacén propio con una fuente externa de confianza (*Source of Truth*) para cuadrar el balance.

### ¡IMPORTANTE! ⚠️
Recuerda **justificar tus decisiones**. YO TU PROFESOR evaluará *por qué* escogiste esa estrategia basada en la teoría, no solo si el código de Pandas corre sin errores.

In [ ]:
import pandas as pd
import hashlib

# --- CÓDIGO DE GENERACIÓN DE DATOS (NO MODIFICAR) ---

# Dataset 1: Ingesta de Almacén (Para validar perímetro)
data_ingesta = {
    'order_id': ['O-001', 'O-002', 'O-003', 'O-004', 'O-005'],
    'product': ['Laptop', 'Mouse', 'Teclado', None, 'Monitor'],
    'quantity': [1, -5, 2, 1, 0], # Hay errores obvios de validación aquí
    'price': [1200.0, 25.0, 45.0, 10.0, 300.0]
}
df_ingesta = pd.DataFrame(data_ingesta)

# Dataset 2: Archivo con Checksum MD5 incrustado (Para integridad de transmisión)
# Simularemos que el proveedor nos envió un hash en metadatos para toda la cadena de texto de la transacción.
data_transito = {
    'tx_id': ['TX-1', 'TX-2', 'TX-3'],
    'data_payload': ['userA:500:OK', 'userB:150:OK', 'userC:999:ERROR'],
    'expected_md5': [
        hashlib.md5('userA:500:OK'.encode()).hexdigest(),
        hashlib.md5('userB:150:OK'.encode()).hexdigest(), # Este hash cuadra
        hashlib.md5('userC:900:ERROR'.encode()).hexdigest() # El proveedor hizo hash a 900, pero nos llegó 999. ¡CORRUPCIÓN!
    ]
}
df_transito = pd.DataFrame(data_transito)

# Dataset 3: Reconciliación 
# Supongamos compras de la web vs cobros reales en el banco (Stripe)
data_web = pd.DataFrame({
    'cart_id': ['C-01', 'C-02', 'C-03'], 
    'status_web': ['DELIVERED', 'DELIVERED', 'PENDING']
})

data_banco = pd.DataFrame({
    'cart_id': ['C-01', 'C-03', 'C-04'], 
    'status_bank': ['PAID', 'PAID', 'PAID']
})

---
### Ejercicio 1: Validación de Frontera (Schema Boundary) y Cuarentena (Dead-Letter)
*(Ref: UT3 puntos 3.3b y 3.5a)*

**Problema:** Muchos errores nacen en la entrada. Si metes datos donde la `quantity` es negativa o el `product` es Nulo a las tablas analíticas, romperás los dashboards.

**Tarea:** 
1. Escribe condiciones de validación (Ej: `quantity > 0` y `product` distinto de nulo).
2. Separa el DataFrame `df_ingesta` original en **dos partes**:
   *   `df_silver_limpio`: La "zona segura" que superó el chequeo.
   *   `df_cuarentena_dead_letter`: Aquellos registros defectuosos que no pasaron la validación, para poder investigarlos en el futuro sin perderlos.

In [ ]:
# TU CÓDIGO AQUÍ



---
### Ejercicio 2: Uso de Hashes/Checksums para Detectar Corrupción de Contenido
*(Ref: UT3 punto 3.1)*

**Problema:** Una transferencia de red se interrumpió o el disco tuvo un pico magnético (Data Corruption). A simple vista, "todo está ahí".

**Tarea:**
1. Crea una nueva columna en `df_transito` llamada `calculated_md5` donde calcules el hash MD5 de la columna `data_payload`.
   *(Pista: Usa un `apply` de pandas con `hashlib.md5(str(x).encode()).hexdigest()`)*
2. Compara `calculated_md5` con `expected_md5`.
3. Filtra y muestra por pantalla exclusivamente la transacción que esté **Corrupta** (donde el checkusm no cuadre).

In [ ]:
# TU CÓDIGO AQUÍ



---
### Ejercicio 3: Auditorías de Consistencia y Reconciliación (*Source of Truth*)
*(Ref: UT3 puntos 3.4b y 3.4c)*

**Problema:** La base de datos dice una cosa y el banco otra (clásico en distribución eventual).

**Tarea:**
Haz un **outer join** (o merge en Pandas) usando `cart_id` entre `data_web` y `data_banco`. Luego responde mediante código y comentarios:
1. ¿A qué carrito se le ha enviado mercancía (`status_web = DELIVERED`) pero resulta que NO consta como pagado en el banco (no existe en `data_banco`)? ¡Alarma de fraude/pérdida!
2. ¿Qué carrito aparece como pagado en el banco (`PAID`) pero nuestra tienda online ni siquiera sabe que existe (no está en `data_web`)? ¡Alarma de datos huérfanos!

In [ ]:
# TU CÓDIGO AQUÍ

